# 🚀 Modelo de Préstamos V2 - Feature Engineering

Este notebook contiene el entrenamiento de la versión V2 del modelo, implementando 5 nuevas características basadas en conocimiento de dominio financiero.

In [ ]:
### SOLO EJECUTAR PARA USAR EN ARCHIVOS DE DRIVE DESDE COLAB
# from google.colab import drive
# drive.mount("/content/drive")

## 1. Carga de Datos y Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# OJO: La ruta debe coincidir con la ubicación del archivo
df = pd.read_csv("dataset_prestamos_500.csv")

def engineer_features(df):
    df_engineered = df.copy()
    
    # 1. DTI: deuda_total / ingresos_anuales
    df_engineered["dti"] = df_engineered["deuda_total"] / df_engineered["ingresos_anuales"]
    
    # 2. Capacidad de Ahorro: ingresos_anuales - deuda_total
    df_engineered["capacidad_ahorro"] = df_engineered["ingresos_anuales"] - df_engineered["deuda_total"]
    
    # 3. Score Normalizado por Edad: score_crediticio / edad
    df_engineered["score_normalizado_edad"] = df_engineered["score_crediticio"] / df_engineered["edad"]
    
    # 4. Apalancamiento Crítico: 1 si deuda_total > (ingresos_anuales * 0.5) sino 0
    df_engineered["apalancamiento_critico"] = (df_engineered["deuda_total"] > (df_engineered["ingresos_anuales"] * 0.5)).astype(int)
    
    # 5. Multiplicador de Estabilidad: (score_crediticio * edad) / 100
    df_engineered["multiplicador_estabilidad"] = (df_engineered["score_crediticio"] * df_engineered["edad"]) / 100.0

    return df_engineered

df_v2 = engineer_features(df)
print("Total features generadas:", len(df_v2.drop("aprobado", axis=1).columns))
df_v2.head()

## 2. Entrenamiento del Modelo V2

In [ ]:
X = df_v2.drop("aprobado", axis=1)
y = df_v2["aprobado"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("✅ Modelo entrenado con las nuevas características.")

## 3. Evaluación y Matriz de Confusión

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Reporte de Clasificación V2:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Rechazado", "Aprobado"],
            yticklabels=["Rechazado", "Aprobado"])
plt.xlabel("Predicción")
plt.ylabel("Realidad")
plt.title("Matriz de Confusión V2")
plt.show()

## 4. Importancia de Variables (Observar el impacto del Feature Engineering)

In [ ]:
importancias = model.feature_importances_
nombres_variables = X.columns

df_importancia = pd.DataFrame({"Variable": nombres_variables, "Importancia": importancias})
df_importancia = df_importancia.sort_values(by="Importancia", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x="Importancia", y="Variable", data=df_importancia, palette="magma", hue="Variable", legend=False)
plt.title("¿Qué factores pesan más en el Modelo V2?")
plt.xlabel("Puntaje de Importancia")
plt.ylabel("Variable")
plt.show()

## 5. Exportar Modelo

In [ ]:
joblib.dump(model, "modelo_prestamos_v2.joblib")
print("💾 Modelo exportado como modelo_prestamos_v2.joblib")